# Profiling pilot — Tiến — Google Colab NVIDIA L4

Máy đo chính thức: **L4**, theo Huy. Notebook này chuẩn bị dụng cụ; **chưa có kết quả PatchCore thật**.

1. Chọn **Runtime → Change runtime type → GPU → L4** nếu tài khoản được cấp L4.
2. Chạy các ô chuẩn bị và tự kiểm; xác nhận GPU thực tế ở ô kiểm L4.
3. Khi Phúc giao baseline, làm theo đúng môi trường/commit của Phúc, điền adapter và config.
4. Chạy full/random/greedy cùng ảnh/thứ tự, batch, máy và ranh giới; tải log về trước khi runtime mất.

Nếu Colab cấp T4/A100/CPU thay L4, dừng phép đo chính thức và hỏi Huy. Không thay thiết bị âm thầm.
Không upload dữ liệu, bank hoặc token vào GitHub công khai. Nơi lưu chung chờ nhóm thống nhất.


In [ ]:
from pathlib import Path
import subprocess
import sys
import json
from datetime import datetime, timezone

REPO_URL = 'https://github.com/TienMoi79/robust-memory-ad.git'
BRANCH = 'profiling/pilot'
ROOT = Path('/content/robust-memory-ad')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(ROOT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(ROOT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote.removesuffix('.git') != REPO_URL.removesuffix('.git'):
        raise RuntimeError('Existing folder has a different remote; inspect before continuing')
    branch = subprocess.check_output(['git', '-C', str(ROOT), 'branch', '--show-current'], text=True).strip()
    if branch != BRANCH:
        raise RuntimeError('Existing checkout is on another branch; inspect before continuing')
import os
os.chdir(ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run(['git', 'status', '--short'], check=True)
print('Existing checkout is kept as-is; record this commit with logs.')


In [ ]:
# Only the profiling dependency; install Phuc's baseline environment separately.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-profiling.txt'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Smoke check CPU — kiểm dụng cụ, không phải benchmark mô hình

Ô này dùng công việc giả lập CPU để kiểm log. Không đưa các số ms vào bảng PatchCore/L4.


In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
smoke_output = ROOT / 'outputs' / ('smoke-colab-cpu-' + stamp)
subprocess.run([sys.executable, '-m', 'scripts.profile', '--smoke', '--output', str(smoke_output)], check=True)
print('Smoke logs:', smoke_output)


## Xác nhận runtime L4

Đây là kiểm môi trường, chưa đo model. Môi trường CPU trên máy Windows của Tiến không thay cho ô này.
Nếu môi trường Phúc yêu cầu phiên bản torch/FAISS cụ thể, cài theo bàn giao của Phúc rồi chạy lại ô này.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable. Select L4 GPU runtime; no CPU fallback.')
gpu_name = torch.cuda.get_device_name(0)
if 'L4' not in gpu_name.upper().split():
    raise RuntimeError(f'Expected NVIDIA L4; received {gpu_name}')
torch.cuda.synchronize(0)
print('GPU:', gpu_name, '| torch:', torch.__version__, '| CUDA:', torch.version.cuda)
subprocess.run(['nvidia-smi'], check=True)
from scripts.profile import environment, write_json
runtime_info = ROOT / 'outputs' / ('l4-runtime-' + stamp)
runtime_info.mkdir(parents=True, exist_ok=False)
write_json(runtime_info / 'environment.json', environment())
print('Runtime evidence:', runtime_info)


## Chờ Phúc: môi trường và pipeline thật

Nhận commit, lệnh cài và lệnh suy luận, full/random/greedy bank, trọng số, ảnh và điểm chuẩn. Random/greedy cùng số patch. Lưu bank/data ở `/content` hoặc Drive theo nơi nhóm chốt; không tạo link public tự động.

Điền `examples/adapter_template.py`, sao chép `examples/pilot-l4.template.json` thành config riêng; mọi đường dẫn tương đối với file config. Adapter phải kiểm commit thật và đưa model về eval/inference. Chốt ranh giới và tolerance trước khi đo.

**Chi phí chọn coreset:** cần ma trận train features sẵn; dùng `measure_selection` như hướng dẫn trong `docs/profiling.md`. Không tính cả extraction train hoặc build index vào chi phí chọn nếu chưa thống nhất.

**Peak VRAM:** bộ đo hiện có bộ đếm PyTorch; FAISS ngoài allocator chưa có peak cấp tiến trình. Cần báo thiếu và thống nhất công cụ với Huy, không gọi PyTorch peak là tổng VRAM.


In [ ]:
# Set only after the handoff and adapter are complete. This cell otherwise does no model work.
READY_TO_PROFILE = False
CONFIG_PATH = None  # e.g. ROOT / 'configs/pilot/pilot01-greedy-s0.json'
ADAPTER_FACTORY = 'examples.adapter_template:create_adapter'

if not READY_TO_PROFILE:
    print('Pending Phuc baseline and confirmed timing boundary; no model results produced.')
else:
    if CONFIG_PATH is None:
        raise ValueError('Set CONFIG_PATH to the completed configuration')
    cfg = json.loads(Path(CONFIG_PATH).read_text(encoding='utf-8'))
    if not cfg.get('require_l4') or cfg.get('device') != 'cuda:0':
        raise ValueError('Official config must require L4 on cuda:0')
    output = ROOT / 'outputs' / cfg['measurement_id']
    subprocess.run([sys.executable, '-m', 'scripts.profile', '--config', str(CONFIG_PATH),
                    '--adapter', ADAPTER_FACTORY, '--output', str(output)], check=True)
    print((output / 'efficiency.csv').read_text())
    print('Review warmup.csv, timings.csv, stability.json, memory.json and status.json before accepting.')


## Bàn giao log

Mỗi lần đo lại dùng measurement_id mới, giữ run_id baseline để ghép quality.csv. Kiểm status=complete; p50 ≤ p95; n đúng; không xóa outlier. Nếu p95 hai nửa dao động, kiểm dãy gốc rồi chạy lại cùng ca/tăng số mẫu và ghi lý do.

Tải toàn bộ outputs trước khi Colab ngắt phiên. Tệp ZIP có thể lớn nếu bạn tự đặt bank vào outputs; nên giữ bank ở artifacts và theo nơi lưu nhóm thống nhất. Mặc định outputs chỉ chứa log nhỏ của bộ đo.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/profiling-logs-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
                              'zip', root_dir=ROOT, base_dir='outputs')
files.download(archive)
